# ResNet-18 — Supplementary Tables S13 & S14 follow-up (random split and MinHash-grouped split, GaussianNoise removed)

**Why this notebook exists.** The main no-GaussianNoise rerun (`ResNet18_full_reverification_no_gaussian_noise.ipynb`)
reruns ResNet-18 under the CD-HIT-97%-grouped split used everywhere else in this study. Two supplementary tables
need ResNet-18 numbers under two *other* split methodologies that notebook never touches:

- **Supplementary Table S13** (random-split leakage ablation) needs ResNet-18 trained with an ordinary random
  split that ignores clustering entirely, at 300 bp and full length, so it can be set against the already-known
  CD-HIT-97% value from the main rerun (300 bp: 0.753, full length: 0.976 — Supplementary Table S4).
- **Supplementary Table S14** (MinHash-grouped split versus CD-HIT-97% split) needs ResNet-18 trained with the
  dataset's built-in MinHash near-duplicate clusters (the `cluster` column in `prepared_sequences_clean.csv`,
  Jaccard threshold 0.90) as the grouping variable, at full length only, again set against the same CD-HIT-97%
  full-length value (0.976).

This notebook is deliberately small: **9 ResNet-18 fits in total** (6 for S13: 300 bp + full length x 3 seeds;
3 for S14: full length x 3 seeds), all using the exact same architecture, training config and TF-IDF 6-mer
encoding as the main no-GaussianNoise rerun — only the train/test split logic differs. No GaussianNoise layer
anywhere, consistent with the rest of this study.

**GPU needed.** Yes. **Runtime.** 9 fits at roughly the same per-fit cost as the main rerun (10-65 min each
depending on early stopping) — realistically 1.5-10 GPU-hours, comfortably inside one Kaggle session.

**How to run.** Edit `DATA_PATH` in Section 2 if your dataset path differs, then Run All. Each section saves
incrementally and skips any `(split, length, seed)` combination it finds already in its own CSV, so a restart
mid-run just picks up where it left off — no separate resume-bridge cells needed for a notebook this small.

**What to send back.** Two CSVs: `resnet18_S13_randomsplit_raw.csv` and `resnet18_S14_minhashsplit_raw.csv`,
both written to `/kaggle/working/`.

### 1. Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, random
import numpy as np, pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, matthews_corrcoef, balanced_accuracy_score, accuracy_score

import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, ReLU, MaxPooling1D,
                                      SpatialDropout1D, Add, GlobalAveragePooling1D, Dropout,
                                      Dense, Layer)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

print("TF:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

### 2. Config

In [ ]:
DATA_PATH = "/kaggle/input/datasets/yashrajtheking/msc-work-final/prepared_sequences_clean.csv"  # <-- EDIT if your path differs

SEEDS          = [0, 1, 2]       # same three seeds used throughout this study
LENGTHS_S13    = [300, None]     # S13: 300 bp and full length; None = full length
EPOCHS         = 120
BATCH          = 48
OUT            = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

### 3. Load data

In [ ]:
df = pd.read_csv(DATA_PATH)
le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
print(f"{len(df)} seqs | {n_cls} classes: {CLASSES}")

assert "cluster" in df.columns, "expected a MinHash 'cluster' column in prepared_sequences_clean.csv"
grp_minhash = df["cluster"].values
print(f"MinHash near-duplicate clusters (Jaccard >= 0.90): {len(set(grp_minhash))} clusters, "
      f"largest = {pd.Series(grp_minhash).value_counts().iloc[0]} sequences")

### 4. Shared helpers: splits, fragmentation, TF-IDF 6-mer encoding, metrics

Fragment sampling uses `rng.integers(0, len(seq) - L + 1)` — the convention used throughout this study (includes every valid start position) — matching the main no-GaussianNoise rerun exactly.

In [ ]:
def split_random(X, y, seed):
    # ordinary random split: completely ignores any clustering/grouping -- this is the
    # "random split" arm needed for Supplementary Table S13
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_idx, te_idx = next(skf.split(X, y))
    skf2 = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)
    tr2, va2 = next(skf2.split(X[tr_idx], y[tr_idx]))
    return tr_idx[tr2], tr_idx[va2], te_idx

def split_grouped(X, y, grp, seed):
    # cluster-disjoint split for any grouping variable -- used here with the MinHash
    # `cluster` column for Supplementary Table S14 (the CD-HIT-97% arm itself is not
    # rerun here; its value is already known from the main no-GaussianNoise rerun)
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te

def fragment(seqs, L, rng):
    if L is None: return list(seqs)
    out = []
    for s in seqs:
        if len(s) <= L: out.append(s)
        else:
            st = int(rng.integers(0, len(s) - L + 1))
            out.append(s[st:st+L])
    return out

def kmers(seq, k=6): return [seq[i:i+k] for i in range(len(seq)-k+1)]

def make_tfidf(train, k=6, max_features=512):
    vec = TfidfVectorizer(max_features=max_features)
    vec.fit([" ".join(kmers(s,k)) for s in train])
    return lambda seqs: vec.transform([" ".join(kmers(s,k)) for s in seqs]).toarray().astype(np.float32)

def score(y_true, y_pred):
    return dict(f1=round(f1_score(y_true, y_pred, average="macro"),4),
                mcc=round(matthews_corrcoef(y_true, y_pred),4),
                bal_acc=round(balanced_accuracy_score(y_true, y_pred),4),
                acc=round(accuracy_score(y_true, y_pred),4))

### 5. ResNet-18 (1D), GaussianNoise removed — identical architecture and training config to the main no-GaussianNoise rerun

In [ ]:
class StochasticDepth(Layer):
    def __init__(self, p=0.85, **kw): super().__init__(**kw); self.p = p
    def call(self, x, training=None):
        if not training: return x
        r = tf.random.uniform([tf.shape(x)[0],1,1]); return x*tf.floor(self.p+r)/self.p
    def get_config(self): return {**super().get_config(), "p": self.p}

def res_block(x, f, stride=1, sp=0.85, dr=0.1):
    sc = x
    x = Conv1D(f,3,strides=stride,padding="same",use_bias=False)(x)
    x = ReLU()(BatchNormalization()(x)); x = SpatialDropout1D(dr)(x)
    x = BatchNormalization()(Conv1D(f,3,padding="same",use_bias=False)(x))
    if stride!=1 or sc.shape[-1]!=f:
        sc = BatchNormalization()(Conv1D(f,1,strides=stride,padding="same",use_bias=False)(sc))
    return ReLU()(Add()([StochasticDepth(sp)(x), sc]))

def build_resnet18_no_noise(L, n):
    inp = Input((L,1))
    x = inp                                              # <-- no GaussianNoise layer, same as the main rerun
    x = ReLU()(BatchNormalization()(Conv1D(32,7,strides=2,padding="same",use_bias=False)(x)))
    x = MaxPooling1D(3,strides=2,padding="same")(x)
    for ff,sp in [(32,.95),(32,.92)]:   x = res_block(x,ff,1,sp)
    for ff,sp in [(64,.88),(64,.85)]:   x = res_block(x,ff,2 if ff!=x.shape[-1] else 1,sp)
    for ff,sp in [(128,.80),(128,.78)]: x = res_block(x,ff,2 if ff!=x.shape[-1] else 1,sp)
    for ff,sp in [(256,.75),(256,.72)]: x = res_block(x,ff,2 if ff!=x.shape[-1] else 1,sp)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    m = Model(inp, Dense(n,activation="softmax")(x))
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def train_resnet(ftr, fva, fte, ytr, yva, cw):
    sc = StandardScaler()                                 # fit on the training fold only
    a = sc.fit_transform(ftr)[...,None]; b = sc.transform(fva)[...,None]; c = sc.transform(fte)[...,None]
    tf.keras.backend.clear_session()
    m = build_resnet18_no_noise(a.shape[1], n_cls)
    m.fit(a, to_categorical(ytr,n_cls), validation_data=(b,to_categorical(yva,n_cls)),
          epochs=EPOCHS, batch_size=BATCH, class_weight=cw, shuffle=True, verbose=0,
          callbacks=[EarlyStopping("val_loss",patience=15,restore_best_weights=True),
                     ReduceLROnPlateau("val_loss",factor=0.5,patience=6,min_lr=1e-6)])
    return m.predict(c, verbose=0).argmax(1)

## Section S13 — Random split (no grouping), 300 bp and full length, 3 seeds

Pairs against the already-known CD-HIT-97% values (300 bp: 0.753, full length: 0.976) from Supplementary Table S4 of the main no-GaussianNoise rerun.

In [ ]:
S13_RAW = f"{OUT}/resnet18_S13_randomsplit_raw.csv"
S13_rows = pd.read_csv(S13_RAW).to_dict("records") if os.path.exists(S13_RAW) else []
S13_done = {(r["length"], r["seed"]) for r in S13_rows}
print(f"[S13] resuming: {len(S13_done)} (length, seed) combinations already done.")

for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tr, va, te = split_random(X_all, y_all, seed)
    ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))

    for L in LENGTHS_S13:
        label = "full" if L is None else str(L)
        if (label, seed) in S13_done:
            print(f"[S13][skip] length={label} seed={seed} already done"); continue

        rng = np.random.default_rng(seed*100 + (L or 9999) + 13000)
        Ftr = fragment(X_all[tr], L, rng); Fva = fragment(X_all[va], L, rng); Fte = fragment(X_all[te], L, rng)

        tfidf = make_tfidf(Ftr)
        Ttr, Tva, Tte = tfidf(Ftr), tfidf(Fva), tfidf(Fte)

        pred = train_resnet(Ttr, Tva, Tte, ytr, yva, cw)

        S13_rows.append({"split":"random","length":label,"length_num":(L or 3000),"seed":seed,
                          **score(yte, pred)})
        pd.DataFrame(S13_rows).to_csv(S13_RAW, index=False)
        print(f"[S13][seed {seed}] L={label:>4}  random-split f1={S13_rows[-1]['f1']}  bal_acc={S13_rows[-1]['bal_acc']}")

S13_summ = (pd.DataFrame(S13_rows).groupby(["length","length_num"])[["f1","mcc","bal_acc","acc"]]
              .agg(["mean","std"]).round(4))
S13_summ.columns = [f"{m}_{s}" for m,s in S13_summ.columns]
S13_summ = S13_summ.reset_index().sort_values("length_num")
S13_summ.to_csv(f"{OUT}/resnet18_S13_randomsplit_summary.csv", index=False)
print(S13_summ.to_string(index=False))

## Section S14 — MinHash-grouped split, full length only, 3 seeds

Uses the dataset's built-in `cluster` column (MinHash near-duplicate grouping, Jaccard >= 0.90) as the grouping variable for `StratifiedGroupKFold`, otherwise identical to the CD-HIT-97% split logic. Pairs against the already-known CD-HIT-97% full-length value (0.976, Supplementary Table S4).

In [ ]:
S14_RAW = f"{OUT}/resnet18_S14_minhashsplit_raw.csv"
S14_rows = pd.read_csv(S14_RAW).to_dict("records") if os.path.exists(S14_RAW) else []
S14_done = {r["seed"] for r in S14_rows}
print(f"[S14] resuming: {len(S14_done)} seed(s) already done.")

for seed in SEEDS:
    if seed in S14_done:
        print(f"[S14][skip] seed={seed} already done"); continue

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tr, va, te = split_grouped(X_all, y_all, grp_minhash, seed)
    ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))

    Ftr, Fva, Fte = list(X_all[tr]), list(X_all[va]), list(X_all[te])   # full length -> no fragmentation
    tfidf = make_tfidf(Ftr)
    Ttr, Tva, Tte = tfidf(Ftr), tfidf(Fva), tfidf(Fte)

    pred = train_resnet(Ttr, Tva, Tte, ytr, yva, cw)

    S14_rows.append({"split":"minhash_grouped","length":"full","length_num":3000,"seed":seed,
                      **score(yte, pred)})
    pd.DataFrame(S14_rows).to_csv(S14_RAW, index=False)
    print(f"[S14][seed {seed}] minhash-grouped f1={S14_rows[-1]['f1']}  bal_acc={S14_rows[-1]['bal_acc']}")

S14_summ = pd.DataFrame(S14_rows)[["f1","mcc","bal_acc","acc"]].agg(["mean","std"]).round(4)
S14_summ.to_csv(f"{OUT}/resnet18_S14_minhashsplit_summary.csv")
print(S14_summ.to_string())

### 6. Zip outputs for download

In [ ]:
!cd /kaggle/working && zip -r resnet18_s13_s14_outputs.zip \
    resnet18_S13_randomsplit_raw.csv resnet18_S13_randomsplit_summary.csv \
    resnet18_S14_minhashsplit_raw.csv resnet18_S14_minhashsplit_summary.csv
print("zipped. Download resnet18_s13_s14_outputs.zip from the Kaggle Output tab, "
      "or just the four CSVs individually -- either works.")